# Catalog Population

Select an existing catalog below. Its `creation-settings.json` snapshot is authoritative for population; this notebook does not read the editable `configs/params-config.json`. The same shared workflow is available through `stormhub populate`, `stormhub resume`, and `stormhub export-dss`; see the [user guide](../docs/source/user_guide.rst) and [rerun runbook](../docs/source/lwi_geometry_rerun.rst). Run discovery separately for 24, 48, and 72 hours.


In [ ]:
from datetime import datetime
from pathlib import Path

from stormhub.logger import initialize_logger
from stormhub.met.catalog_setup import load_catalog_settings
from stormhub.met.storm_catalog import (
    StormCatalog,
    create_normal_precip,
    stac_to_parquet,
)
from stormhub.met.catalog_population import populate_catalog, resume_catalog, export_catalog_dss

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

# Select the catalog to populate independently of the next catalog's creation config.
CATALOG_ID = "lwi-region3-geometry-v2"
CATALOG_DIR = REPO_ROOT / "catalogs" / CATALOG_ID
CATALOG_FILE = CATALOG_DIR / "catalog.json"
settings = load_catalog_settings(CATALOG_DIR / "creation-settings.json")
if settings["catalog_id"] != CATALOG_ID:
    raise ValueError("The creation-settings snapshot does not belong to the selected catalog.")
run_settings = settings["params"]

CATALOG_FILE


## Population Parameters

`SMOKE_TEST = True` limits the run to the dates in `SPECIFIC_DATES`. Set it to `False` for the full configured date range.

In [ ]:
# All Collection Args

# Collection Args
STORM_DURATION_HOURS = run_settings["storm_duration_hours"] # rolling accumulation window, e.g. 48hr-events
NUM_WORKERS = run_settings["num_workers"] # number of parallel workers for event processing

# Development controls
SMOKE_TEST = False # limits events to specific dates quick testing
SPECIFIC_DATES = [datetime(2017, 8, 27, 6), 
                  datetime(2019, 9, 17, 6), 
                  datetime(2016, 3, 8, 18), 
                  datetime(2016, 8, 11, 12)] # targeted dates for smoke tests or reruns. this is buggy, currently only limits to the specific dates entered and not a range
WITH_TRACEBACK = True # detailed debugging output

# Optional derived products
RUN_DSS_EXPORT = True # adds DSS files to storm items
DSS_OUTPUT_MODES = tuple(run_settings["dss_output_modes"]) # source preserves event location; target moves it to the watershed
DSS_ITEM_IDS = ["1"] # set to a list such as ["1"] for a targeted export
RUN_NORMAL_PRECIP = False # builds annual max/normal precip grids
RUN_GEOPARQUET_EXPORT = True # writes EDA-friendly parquet table

# Fresh population refuses an existing duration directory. For a replacement,
# prepare a new catalog; use resume only for a recorded interrupted full search.
OVERWRITE_DSS = False  # Explicit opt-in to replace the selected DSS outputs.

params = {
    **run_settings,
    "storm_duration_hours": STORM_DURATION_HOURS,
    "num_workers": NUM_WORKERS,
    "smoke_test": SMOKE_TEST,
    "dss_output_modes": DSS_OUTPUT_MODES,
    "dss_item_ids": DSS_ITEM_IDS,
}
params

## Inputs

Search dates, event count, precipitation threshold, search interval, executor mode, output resolution, target buffer, and valid-region choice come from the frozen `run_settings`. Prepare a new catalog generation to change these scientific defaults.

| Control | Purpose |
| --- | --- |
| `STORM_DURATION_HOURS` | Duration and collection ID, such as `48hr-events`. |
| `NUM_WORKERS` | Worker-count override; saved executor mode is retained. |
| `SMOKE_TEST`, `SPECIFIC_DATES` | Use only these exact UTC start times when smoke testing. |
| `WITH_TRACEBACK` | Include diagnostic tracebacks. |
| `RUN_DSS_EXPORT` | Enable the separate DSS export cell after population review. |
| `DSS_OUTPUT_MODES` | Source, target, or both; defaults from the frozen snapshot. |
| `DSS_ITEM_IDS` | Explicit IDs; `None` intentionally selects the whole collection. |
| `OVERWRITE_DSS` | Explicitly replace selected existing DSS products. Preserve a backup first. |
| `RUN_NORMAL_PRECIP` | Build optional annual-max/normal precipitation grids. |
| `RUN_GEOPARQUET_EXPORT` | Export the resulting collection to GeoParquet. |


## Load Catalog

In [ ]:
if not CATALOG_FILE.exists():
    raise FileNotFoundError(f"Create the base catalog first: {CATALOG_FILE}")

initialize_logger()
storm_catalog = StormCatalog.from_file(str(CATALOG_FILE))
if storm_catalog.id != CATALOG_ID:
    raise ValueError("The STAC catalog ID does not match the selected catalog and its creation settings.")
storm_catalog.id, storm_catalog.spm.catalog_file


## Create a fresh event collection

Set `SMOKE_TEST = False` for the full frozen date range. Fresh population refuses any existing duration workspace, including a smoke collection. Prepare a new catalog generation for a replacement/full search after a smoke run. The shared function writes `population-settings.json` before starting compute.


In [ ]:
storm_collection = populate_catalog(
    CATALOG_FILE,
    duration_hours=STORM_DURATION_HOURS,
    num_workers=NUM_WORKERS,
    specific_dates=SPECIFIC_DATES if SMOKE_TEST else None,
    with_tb=WITH_TRACEBACK,
)

storm_collection.id

## Resume missing dates (alternative to discovery)

Use this cell only for a recorded interrupted full search with unchanged settings and geometry, partial statistics, and no Items or DSS yet. Smoke searches, legacy runs without `population-settings.json`, and interruptions after Item creation require separate inspection. Do not run discovery and resume as routine sequential steps.


In [ ]:
RUN_RESUME = False  # Alternative to fresh population, for an interrupted full search.
if RUN_RESUME:
    storm_collection = resume_catalog(
        CATALOG_FILE,
        duration_hours=STORM_DURATION_HOURS,
        num_workers=NUM_WORKERS,
        with_tb=WITH_TRACEBACK,
    )

## Optional DSS assets

After reviewing the population, enable `RUN_DSS_EXPORT` and select `DSS_ITEM_IDS=["1"]` for a smoke export. Choose explicit remaining IDs to retain successful products. `None` selects the entire collection. Existing selected DSS files are refused unless `OVERWRITE_DSS=True`; replacement is not atomic. `export_catalog_dss(..., dry_run=True)` inspects the selection without exporting.


In [ ]:
dss_export_result = None

if RUN_DSS_EXPORT:
    dss_export_result = export_catalog_dss(
        CATALOG_FILE,
        duration_hours=STORM_DURATION_HOURS,
        output_modes=DSS_OUTPUT_MODES,
        item_ids=DSS_ITEM_IDS,
        all_items=DSS_ITEM_IDS is None,
        overwrite=OVERWRITE_DSS,
    )
    if dss_export_result["failed_count"]:
        raise RuntimeError(
            f"DSS export failed for {dss_export_result['failed_count']} item(s): "
            f"{dss_export_result['failed_items']}"
        )
else:
    print("RUN_DSS_EXPORT is False; skipping DSS export.")

dss_export_result

### DSS Pipeline Enhancements

## Optional Normal Precipitation Grid

This computes annual maximum grids and averages them into a normal precipitation GeoTIFF. Consider testing a short year range before running the full 1980-2024 period.

In [ ]:
if RUN_NORMAL_PRECIP:
    normal_dir = CATALOG_DIR / "normal_precip"
    normal_dir.mkdir(parents=True, exist_ok=True)

    create_normal_precip(
        start_year=1980,
        end_year=2024,
        catalog=storm_catalog,
        storm_duration_hours=STORM_DURATION_HOURS,
        every_n_hours=24,
        months=None,
        ams_zarr_path=str(normal_dir / "ams_grids.zarr"),
        normal_precip_grid_path=str(normal_dir / "normalized_precip.tif"),
    )
else:
    print("RUN_NORMAL_PRECIP is False; skipping normal precipitation grid.")

## Optional GeoParquet Export

GeoParquet gives you a compact table for EDA after the STAC item JSON has been created.

In [ ]:
if RUN_GEOPARQUET_EXPORT and storm_collection is not None:
    parquet_path = CATALOG_DIR / storm_collection.id / "all-items.parquet"
    stac_to_parquet(storm_collection, parquet_file=str(parquet_path))
    print(parquet_path)
else:
    print("Skipping GeoParquet export.")

## Serve The Populated Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3-geometry-v2 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or the STAC Browser link shown by the directory listing.